# Scoping the answer space

**Question:** what should the trained model predict? Options from the problem statement:
- **Extractive:** pick the answer out of the extracted table (a cell, a number or a name).
- **Binned classification:** predict a class, such as a numeric range.

This notebook looks at the ChartQA answers and questions to decide. The helper code is in `utils/answer_space.py`.
Types are assigned with simple rules (the answer's form, and keywords in the question), so the numbers are approximate.

## 1. What the answers look like (all 28,299 train rows)

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "data_exploration" else Path.cwd()
sys.path.insert(0, str(ROOT / "utils"))
import answer_space as A

train = A.load_split("train-*.parquet")
print("train rows:", len(train))
print()
print("answer type, % of rows (group 0 = human-written, 1 = machine-generated)")
overall = (train["atype"].value_counts(normalize=True) * 100).rename("all")
by_group = pd.crosstab(train["atype"], train["human_or_machine"], normalize="columns") * 100
print(pd.concat([overall, by_group], axis=1).round(1).to_string())

train rows: 28299

answer type, % of rows (group 0 = human-written, 1 = machine-generated)
         all     0     1
atype                   
number  80.7  68.9  84.9
text    16.6  20.7  15.1
yes/no   2.1   8.0   0.0
list     0.6   2.3   0.0


## 2. What the questions ask the model to do

In [2]:
print("question operation, % of rows")
overall = (train["op"].value_counts(normalize=True) * 100).rename("all")
by_group = pd.crosstab(train["op"], train["human_or_machine"], normalize="columns") * 100
print(pd.concat([overall, by_group], axis=1).round(1).to_string())
print()
print("answer type inside each operation, % of that operation")
print((pd.crosstab(train["op"], train["atype"], normalize="index") * 100).round(1).to_string())

question operation, % of rows
                                               all     0     1
op                                                            
look up a value or name                       48.5  28.9  55.4
how many (a count or a value)                 17.7   8.9  20.8
compute (sum, difference, average, ratio...)  17.3  35.6  10.8
pick highest/lowest (superlative)             14.5  18.6  13.0
yes/no (compare or check)                      2.1   8.0   0.0

answer type inside each operation, % of that operation
atype                                         list  number  text  yes/no
op                                                                      
compute (sum, difference, average, ratio...)   0.7    94.2   5.1     0.0
how many (a count or a value)                  0.0    99.9   0.0     0.0
look up a value or name                        0.7    86.1  13.2     0.0
pick highest/lowest (superlative)              0.9    34.7  64.3     0.0
yes/no (compare or check)          

## 3. Numeric answers: could they be binned?

With the 5% tolerance of relaxed accuracy, a bin classifier needs bins about 10% wide (on a log scale).

In [3]:
import math

num = train[train["atype"] == "number"].copy()
num["val"] = num["labels"].apply(lambda l: A.to_number(l[0]))

print("numeric answers:", len(num))
print("whole numbers:      ", round((num["val"] == num["val"].round()).mean() * 100, 1), "%")
print("between 0 and 100:  ", round(num["val"].between(0, 100).mean() * 100, 1), "%")

magnitude = np.floor(np.log10(num["val"].abs().replace(0, np.nan))).dropna().astype(int)
print()
print("size of the answer (power of ten), % of numeric answers")
print((magnitude.value_counts(normalize=True).sort_index() * 100).round(1).to_string())

positive = num[num["val"] > 0]["val"]
bin_id = np.floor(np.log(positive) / math.log(1.10)).astype(int)
counts = bin_id.value_counts()
print()
print("distinct 10%-wide bins in use:", len(counts))
for k in (20, 50, 100, 200):
    print(f"top {k:>3} bins cover {counts.head(k).sum() / len(positive) * 100:.1f}% of numeric answers")
exact = num["labels"].apply(lambda l: l[0]).value_counts()
print(f"distinct exact numeric answers: {len(exact)} (top 50 cover {exact.head(50).sum() / len(num) * 100:.1f}%)")

numeric answers: 22847
whole numbers:       49.9 %
between 0 and 100:   58.0 %

size of the answer (power of ten), % of numeric answers
val
-3     0.0
-2     0.5
-1     3.1
 0    18.1
 1    36.1
 2    18.1
 3    14.7
 4     5.1
 5     2.4
 6     1.4
 7     0.3
 8     0.1
 9     0.0

distinct 10%-wide bins in use: 253
top  20 bins cover 38.1% of numeric answers
top  50 bins cover 66.4% of numeric answers
top 100 bins cover 88.2% of numeric answers
top 200 bins cover 99.5% of numeric answers
distinct exact numeric answers: 9675 (top 50 cover 17.0%)


## 4. Text answers: could they be a fixed set of classes?

In [4]:
text = train[train["atype"] == "text"]
text_counts = text["labels"].apply(lambda l: l[0].strip().lower()).value_counts()
print("text answers:", len(text), "| distinct:", len(text_counts))
for k in (50, 200, 1000):
    print(f"top {k:>4} answers cover {text_counts.head(k).sum() / len(text) * 100:.1f}%")
print("most common:", text_counts.head(10).to_dict())

text answers: 4687 | distinct: 2604
top   50 answers cover 24.0%
top  200 answers cover 39.8%
top 1000 answers cover 65.8%
most common: {'china': 118, 'germany': 86, 'brazil': 53, 'blue': 47, 'france': 45, 'gray': 42, 'red': 37, 'united states': 36, 'green': 33, 'russia': 31}


## 5. Can the answer be read straight from the table?

Uses the 300-row validation sample and the tables DePlot and UniChart extracted. The check is rough: a text answer must appear in the table, and a number must equal some number in it (also allowing a percent against a fraction).

In [5]:
val = A.load_split("val-*.parquet")
sample = pd.read_csv(ROOT / "results/val_sample_300.csv")
dp_path, uc_path = ROOT / "results/deplot_val_300.jsonl", ROOT / "results/unichart_val_300.jsonl"

if dp_path.exists() and uc_path.exists():
    dp = pd.read_json(dp_path, lines=True, dtype=False).set_index("row_idx")
    uc = pd.read_json(uc_path, lines=True, dtype=False).set_index("row_idx")
    v = val.iloc[sample["row_idx"]].reset_index(drop=True)
    v["row_idx"] = sample["row_idx"]
    v["in DePlot table"] = [A.answer_in_table(l, dp.loc[r, "deplot_table"]) for l, r in zip(v["labels"], v["row_idx"])]
    v["in UniChart table"] = [A.answer_in_table(l, uc.loc[r, "unichart_table"]) for l, r in zip(v["labels"], v["row_idx"])]
    cols = ["in DePlot table", "in UniChart table"]

    print("% of answers readable straight from the table, by answer type")
    print((v.groupby("atype")[cols].mean() * 100).round(1).assign(n=v.groupby("atype").size()).to_string())
    print()
    print("by question operation")
    print((v.groupby("op")[cols].mean() * 100).round(1).assign(n=v.groupby("op").size()).to_string())
    print()
    print("overall:", (v[cols].mean() * 100).round(1).to_dict())
else:
    print("Run utils/run_deplot.py and utils/run_unichart.py first.")

% of answers readable straight from the table, by answer type
        in DePlot table  in UniChart table    n
atype                                          
list                0.0                0.0    3
number             73.2               70.6  231
text               94.6               92.9   56
yes/no             30.0               20.0   10

by question operation
                                              in DePlot table  in UniChart table    n
op                                                                                   
compute (sum, difference, average, ratio...)             37.9               36.2   58
how many (a count or a value)                            67.6               64.9   37
look up a value or name                                  88.9               84.4  135
pick highest/lowest (superlative)                        91.7               93.3   60
yes/no (compare or check)                                30.0               20.0   10

overall: {'in DePlot tab

## 6. Decision

**Extractive first, with two small extra heads. Not binned classification.**

Why not bins or a fixed class list:
- Numeric answers are spread out: nearly 10,000 distinct values, and about 250 bins of 10% width are in use. A classifier can only pick the right bin if it has read the number from the chart, so reading it (extracting it) is the real task.
- Text answers are a long tail: over 2,600 distinct answers (country names, colours, categories), and the 1,000 most common cover only about two thirds. A fixed class list would miss a third of them. In the validation sample, 95% of text answers appear in the extracted table.

Why extractive works:
- Questions that look up a value or pick the highest or lowest (about 63% of questions) have their answer in the table about 90% of the time.

What extractive cannot do, so it needs extra heads:
- **Compute questions** (sum, difference, average, ratio; about 17% of train, and 36% of human-written questions) have their answer in the table only about 38% of the time. They need an operation head that applies sum, difference, average or ratio to numbers picked from the table.
- **"How many" questions** are a mix of counts and looked-up values, so the answer is in the table about two thirds of the time.
- **Yes/no questions** (2.1% of train, all human-written) are a 2-class head.

So the plan: a router that decides the answer kind (extract, compute, yes/no), an extractive picker over the table, a small operation head for computed answers, and a yes/no head.

Ceiling to keep in mind: reading the answer straight from the table covers about 75% of validation answers with DePlot (72% with UniChart). The zero-shot LLM baseline reaches 81% because it can also compute.

Caveats: types come from keyword rules, the in-table check is rough, and section 5 uses only 300 validation questions.